In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# 1. Load Data
train_path = 'C:/Users/PGCP-AI/Downloads/IrrigationKaggle/train.csv'
test_path = 'C:/Users/PGCP-AI/Downloads/IrrigationKaggle/test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# 2. Expert Feature Engineering
def add_features(df):
    df['Moisture_Efficiency'] = df['Soil_Moisture'] / (df['Temperature_C'] + df['Humidity'] + 1)
    df['Rain_Retention'] = df['Rainfall_mm'] / (df['Soil_Moisture'] + 1)
    df['is_arid'] = ((df['Temperature_C'] > 35) & (df['Humidity'] < 40)).astype(int)
    df['is_soaked'] = (df['Soil_Moisture'] > 70).astype(int)
    df['Soil_Stress'] = df.groupby('Soil_Type')['Soil_Moisture'].transform('mean') - df['Soil_Moisture']
    return df

train = add_features(train)
test = add_features(test)

# 3. Handle Categoricals
cat_cols = train.select_dtypes(exclude=[np.number]).columns.tolist()
if 'Irrigation_Need' in cat_cols: cat_cols.remove('Irrigation_Need')

for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = pd.Categorical(test[col], categories=train[col].cat.categories)

# 4. Target Encoding
le = LabelEncoder()
target = le.fit_transform(train['Irrigation_Need'])

X = train.drop(['id', 'Irrigation_Need'], axis=1)
X_test = test.drop(['id'], axis=1)

# 5. Fixed Model Parameters
xgb_params = {
    'n_estimators': 2500,
    'learning_rate': 0.015,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'multi:softprob',
    'num_class': 3,
    'tree_method': 'hist',
    'enable_categorical': True,
    'random_state': 42,
    # MOVED EARLY STOPPING HERE 👇
    'early_stopping_rounds': 50 
}

# GPU Check logic for different XGBoost versions
try:
    import torch
    if torch.cuda.is_available():
        xgb_params['device'] = 'cuda'
except:
    pass

# 6. Training Loop
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
test_preds = np.zeros((len(X_test), 3))

print("Starting training...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, target)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = target[train_idx], target[val_idx]
    
    # Early stopping now monitors the eval_set automatically
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train, 
              eval_set=[(X_val, y_val)], 
              verbose=False)
    
    test_preds += model.predict_proba(X_test) / 10
    
    fold_acc = accuracy_score(y_val, model.predict(X_val))
    print(f"Fold {fold+1} Acc: {fold_acc:.5f}")

# 7. Final Submission
final_labels = le.inverse_transform(np.argmax(test_preds, axis=1))
submission = pd.DataFrame({'id': test['id'], 'Irrigation_Need': final_labels})
submission.to_csv('C:/Users/PGCP-AI/Downloads/IrrigationKaggle/submission.csv', index=False)
print("✅ Submission file saved as 'submission.csv'")


Starting training...
Fold 1 Acc: 0.98451
Fold 2 Acc: 0.98505
Fold 3 Acc: 0.98473
Fold 4 Acc: 0.98476
Fold 5 Acc: 0.98440
Fold 6 Acc: 0.98421
Fold 7 Acc: 0.98490
Fold 8 Acc: 0.98462
Fold 9 Acc: 0.98502
Fold 10 Acc: 0.98483
✅ Submission file saved as 'submission.csv'
